# Ingeniería de software

Hasta ahora el curso se enfocó en que tu código *corra rápido*. Esta notebook se enfoca en algo distinto pero igual de importante: que tu código sea **mantenible, confiable y compartible** — que tú mismo (dentro de 6 meses) y otras personas puedan entenderlo, confiar en que funciona, e instalarlo sin dolores de cabeza.

Un recorrido rápido por:
* Empaquetado (*packaging*)
* Gestión de dependencias
* Documentación
* Control de versiones
* Testing (pruebas de software)
* Distribución (PyPI)

## Antes de empezar: tres palabras que vas a ver todo el tiempo

* **Paquete (*package*):** una carpeta de código Python organizada de una forma estándar, para que se pueda instalar y reutilizar fácilmente — en vez de copiar archivos `.py` sueltos de un proyecto a otro.
* **`pip`:** el programa que usas para instalar paquetes (`pip install numpy`). Ya lo usaste, aunque no lo hayas notado: cuando creaste tu entorno con `environment.yml`, conda instaló varios paquetes, y algunos internamente usan `pip`.
* **PyPI (Python Package Index):** una página web (https://pypi.org) que funciona como una tienda de aplicaciones, pero de paquetes de Python en vez de apps — es de ahí de donde `pip install numpy` descarga NumPy. Cualquier persona puede subir un paquete a PyPI después de registrarse, así que no hay garantía de calidad automática: instalar un paquete de PyPI es confiar en que quien lo subió hizo bien su trabajo.

Con estas tres palabras ya puedes entender la frase clave de esta notebook: **empaquetar tu código significa organizarlo de la forma que `pip` y PyPI esperan, para que cualquiera (incluido tú mismo) lo pueda instalar con un solo comando.**

## Empaquetado de software

Imagina que escribes una función útil para tu tesis, y seis meses después un compañero de laboratorio la necesita. ¿Le mandas el archivo `.py` por WhatsApp? ¿Y si esa función depende de otras 3 funciones que están en otros archivos, y él no sabe en qué orden acomodarlas? Empaquetar tu código resuelve justo ese problema: en vez de mandar archivos sueltos, tu compañero simplemente instala tu paquete y ya tiene todo funcionando.

El ecosistema de Python está pensado desde su base para esto:
* Permite crear un paquete instalable con `pip install`, ya sea desde PyPI o directamente desde una carpeta en tu computadora.
* El paquete declara sus **dependencias** (qué otras librerías necesita) — así, al instalarlo, esas librerías se instalan solas también.
* Guarda metadatos: versión, quién lo escribió, bajo qué licencia se puede usar.
* Puede incluso compilar extensiones en C/C++/Fortran/Cython como parte de la instalación.

Ver ejemplos reales en la carpeta [`examples/setuptools`](../examples/setuptools) de este repositorio.

### El archivo `pyproject.toml`

Cada paquete de Python necesita un archivo llamado `pyproject.toml` en su carpeta raíz — es literalmente la "ficha técnica" del paquete: cómo se llama, qué versión es, quién lo escribió, y qué necesita para funcionar.

Vas a ver que existen varias herramientas relacionadas con esto (`setuptools`, `hatch`, `poetry`, `flit`, `pdm`...). No necesitas memorizarlas: todas hacen básicamente lo mismo (leer el `pyproject.toml` y construir el paquete), son solo distintas implementaciones. En este curso usamos `setuptools`, la más tradicional y todavía la más común.

### Ejemplo de un `pyproject.toml`

```toml
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "helloworld"
version = "0.1"
description = "paquete de ejemplo que imprime hello world"
# dependencies = ["numpy"]
authors = [
  {name = "Nombre Apellido", email = "correo@ejemplo.com"}
]
```

Los puntos importantes para quedarte:
* `name` es el nombre con el que se instalaría (`pip install helloworld`).
* La línea comentada `# dependencies = ["numpy"]` es donde declararías que tu paquete necesita NumPy — si la descomentas, cuando alguien instale tu paquete, NumPy se instala automáticamente si no lo tiene.
* El bloque `[build-system]` le dice a `pip` qué herramienta usar para construir el paquete (en este caso, `setuptools`) — es la parte más "de configuración" y menos importante de entender en detalle por ahora.

### Instalar tu propio paquete

* `pip install .` (ejecutado dentro de la carpeta del paquete) lo construye y lo instala.
* `pip install -e .` lo instala en modo *editable*: en vez de copiar los archivos, le dice a Python "usa directamente los archivos de esta carpeta". Así, si editas tu código después, los cambios se reflejan al instante, sin reinstalar — súper útil mientras estás desarrollando.
* Cuando ya quieres *compartir* tu paquete (no solo usarlo tú), puedes generar un archivo para distribuir con `python -m build`. Esto genera dos tipos de archivo: uno con el código fuente, y otro ya "listo para instalar" (llamado *wheel*) — no necesitas más detalle que eso para este curso.

### Estructura de carpetas de un paquete simple

```
carpeta_raiz_del_proyecto/
    pyproject.toml
    nombre_del_paquete/            # carpeta del paquete
        __init__.py                # inicializa el paquete
        modulo_a.py                # un modulo de python
        modulo_b.py                # otro modulo
```

`__init__.py` es un archivo que le dice a Python "esta carpeta es un paquete" — puede estar vacío, o contener código que se ejecuta cuando alguien hace `import nombre_del_paquete`. Una vez instalado, se importa igual que cualquier librería: `import nombre_del_paquete` o `from nombre_del_paquete import modulo_a`.

### ¿Qué más suele llevar un paquete "completo"?

Además del código, un paquete publicado normalmente incluye:
* **Documentación de usuario:** cómo usar el paquete.
* **Pruebas de software** (*tests*): para comprobar que funciona (lo vas a ver más abajo).
* Archivos de texto estándar que la gente espera encontrar en un repositorio:
  * `README` — qué es el proyecto y cómo empezar a usarlo (ya viste uno: el de este mismo curso).
  * `LICENSE` — bajo qué términos se puede usar/redistribuir el código.
  * `CONTRIBUTING` — cómo contribuir al proyecto.
  * `CHANGELOG` — qué cambió en cada versión.

### Actividad: crea tu propio paquete mínimo

Vamos a crear un paquete de Python real, siguiendo la estructura de arriba. `%%writefile` es un "magic" de Jupyter que escribe el contenido de la celda directamente a un archivo en disco.

**Paso 1.** Crea la estructura de carpetas.

In [1]:
import os
os.makedirs("mi_paquete/mi_paquete", exist_ok=True)

**Paso 2.** Crea el `pyproject.toml` (la ficha técnica del paquete).

In [2]:
%%writefile mi_paquete/pyproject.toml
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "mi_paquete"
version = "0.1"
description = "Mi primer paquete de Python"
authors = [
  {name = "David Israel Atilano Quiroz"}
]

Overwriting mi_paquete/pyproject.toml


**Paso 3.** Crea el código del paquete: una función simple dentro de `__init__.py`.

In [3]:
%%writefile mi_paquete/mi_paquete/__init__.py
def saludar(nombre):
    """
    Saluda a una persona por su nombre.

    Parameters
    ----------
    nombre : str
        El nombre de la persona a saludar.

    Returns
    -------
    str
        El saludo generado.
    """
    return f"Hola, {nombre}! Este saludo viene de un paquete instalado de verdad."


Overwriting mi_paquete/mi_paquete/__init__.py


**Paso 4.** Instálalo. Esta parte se hace en una terminal (no en esta notebook), porque instalar un paquete cambia tu entorno de Python de forma permanente — mejor hacerlo de forma explícita, no "escondido" dentro de una celda:

```bash
cd mi_paquete
pip install -e .
```

**Paso 5.** Compruébalo. Abre una notebook o consola de Python nueva (en el mismo entorno) y ejecuta:

```python
from mi_paquete import saludar
print(saludar("Ana"))
```

Deberías ver el mensaje que escribiste en el `return`. Si editas la función y la vuelves a llamar, el cambio aparece sin reinstalar nada — eso es lo que significa "editable".

**Para pensar:** ¿por qué esto es mejor que copiar y pegar la función `saludar` en cada script nuevo que la necesite?

**Para limpiar:** cuando termines de probarlo, puedes desinstalarlo con `pip uninstall mi_paquete`.

In [4]:
# Paso 5: lo compruebo en un proceso de Python nuevo, desde otra carpeta.
# (Si lo importo aqui mismo, Python encuentra primero la carpeta mi_paquete/
#  del directorio actual y no el paquete instalado; lo explico abajo.)
!cd .. && python -c "from mi_paquete import saludar; print(saludar('Ana'))"

Hola, Ana! Este saludo viene de un paquete instalado de verdad.


**Mis resultados de la actividad del paquete**

- En la terminal entré a `mi_paquete` y corrí `pip install -e .`. Terminó con `Successfully installed mi_paquete-0.1`.
- Primero intenté importar `saludar` directamente en esta notebook y salió un `ImportError`. La causa es que la notebook corre dentro de `notebooks/`, donde también existe la carpeta `mi_paquete/`. Python la encontró primero y la tomó como un paquete vacío (un *namespace package*), en lugar del paquete que instalé.
- Por eso lo comprobé como dice el Paso 5, con un proceso de Python nuevo desde otra carpeta. Ahí sí imprimió `Hola, Ana! Este saludo viene de un paquete instalado de verdad.`

**Para pensar:** empaquetar es mejor que copiar y pegar porque la función vive en un solo lugar. Si encuentro un error o la mejoro, la corrijo una vez y todos los scripts que la usan reciben el cambio. Con copias pegadas tendría varias versiones distintas de la misma función, y no sabría cuál está actualizada. Además, el paquete declara su versión y sus dependencias, así que otra persona lo instala con un solo comando.

## Gestión de dependencias

Manejar las dependencias de varios proyectos de Python en la misma computadora es difícil, por tres razones:
* **Conflictos:** el proyecto A necesita NumPy 1.x, el proyecto B necesita NumPy 2.x, y no puedes tener las dos versiones instaladas "sueltas" al mismo tiempo.
* **Mantenimiento:** actualizar una librería para un proyecto puede romper otro que dependía de la versión vieja.
* **Reproducibilidad:** que el código funcione igual en tu laptop, en la de tu compañero, y en el clúster.

La solución, como ya sabes por la Tarea 1 del notebook 01, es aislar cada proyecto en su propio **entorno virtual**: una "caja" separada de paquetes instalados, que no interfiere con otras cajas.

### `venv`: la versión que trae Python

Ya usaste `conda` para crear el entorno `pyhpc` de este curso. Python también trae su propia herramienta más simple, llamada `venv`, integrada de fábrica (no hace falta instalar nada extra):

```bash
python -m venv myenv        # crea la "caja" en la carpeta myenv/
source myenv/bin/activate   # entra a esa caja
pip install numpy           # instala paquetes solo dentro de esa caja
deactivate                  # sale de la caja
```

**¿Cuál usar, `venv` o `conda`?**
* `conda` (lo que ya usas): puede instalar de todo, incluyendo compiladores y librerías que no son de Python — necesario para este curso (MPI, gcc, etc.).
* `venv`: más simple y liviano, pero solo maneja paquetes de Python. Es una buena opción cuando tu proyecto no necesita nada especial del sistema.

No necesitas elegir uno para siempre — en la práctica vas a encontrarte ambos en distintos proyectos.

## Documentación

### Comentarios

Los *comentarios* empiezan con `#` y son notas dirigidas a quien programa (incluido tu yo del futuro), no a quien va a *usar* tu código:

```python
# TODO: inicializar la lista con valores por defecto
a = []
```

Ponlos donde aclaren algo que no es obvio a simple vista, o para recordar pendientes — evita comentarios que solo repiten lo que el código ya dice (`x = x + 1  # suma 1 a x` no aporta nada).

### Docstrings

Un *docstring* es distinto de un comentario: describe *cómo usar* una función, clase o módulo, y va justo debajo de su definición, entre comillas triples. La diferencia clave: un docstring se puede consultar en cualquier momento con `help()`, un comentario no.

In [5]:
class SimulationData:
    """Clase para representar datos de simulaciones de hidrodinamica"""
    def __init__(self, path):
        """Inicializa un objeto SimulationData desde una ruta dada"""
        self.load_data(path)

    def load_data(self, path):
        """Carga los datos de la simulacion desde la ruta dada"""
        # hacer algo aca
        pass

### Docstrings estilo NumPy

En cómputo científico, el estilo de docstring más usado es el de NumPy: describe explícitamente el tipo y propósito de cada parámetro de entrada (`Parameters`), y qué devuelve la función (`Returns`).

In [6]:
def add(a, b):
    """
    Suma dos numeros.

    Esta funcion suma dos numeros y devuelve el resultado.

    Parameters
    ----------
    a : int o float
        El primer sumando.
    b : int o float
        El segundo sumando.

    Returns
    -------
    int o float
        La suma de a y b.
    """
    return a + b

help(add)

Help on function add in module __main__:

add(a, b)
    Suma dos numeros.
    
    Esta funcion suma dos numeros y devuelve el resultado.
    
    Parameters
    ----------
    a : int o float
        El primer sumando.
    b : int o float
        El segundo sumando.
    
    Returns
    -------
    int o float
        La suma de a y b.



### Actividad: documenta una función que ya escribiste

**Paso 1.** Elige una función que hayas escrito en el notebook 04 (por ejemplo, `midpoints_array` o `center_of_mass_dot`).

**Paso 2.** Escríbela de nuevo en la celda de abajo, reemplazando el docstring de ejemplo por uno completo en estilo NumPy (descripción corta, `Parameters`, `Returns`).

**Paso 3.** Ejecuta la celda y después corre `help()` sobre tu función, para comprobar que el docstring se ve bien.

In [7]:
import numpy as np

def midpoints_array(x):
    """
    Calcula los puntos medios entre elementos consecutivos de un arreglo.

    Para cada par de valores vecinos x[i] y x[i+1] devuelve su promedio,
    usando operaciones vectorizadas de NumPy (sin loops de Python).

    Parameters
    ----------
    x : ndarray
        Arreglo unidimensional de numeros (int o float) con n elementos.

    Returns
    -------
    ndarray
        Arreglo de n-1 elementos, donde el elemento i es (x[i] + x[i+1]) / 2.

    Examples
    --------
    >>> midpoints_array(np.array([0.0, 2.0, 4.0]))
    array([1., 3.])
    """
    return 0.5*(x[:-1] + x[1:])

help(midpoints_array)

Help on function midpoints_array in module __main__:

midpoints_array(x)
    Calcula los puntos medios entre elementos consecutivos de un arreglo.
    
    Para cada par de valores vecinos x[i] y x[i+1] devuelve su promedio,
    usando operaciones vectorizadas de NumPy (sin loops de Python).
    
    Parameters
    ----------
    x : ndarray
        Arreglo unidimensional de numeros (int o float) con n elementos.
    
    Returns
    -------
    ndarray
        Arreglo de n-1 elementos, donde el elemento i es (x[i] + x[i+1]) / 2.
    
    Examples
    --------
    >>> midpoints_array(np.array([0.0, 2.0, 4.0]))
    array([1., 3.])



### Sphinx: generar documentación en HTML automáticamente

Cuando un paquete crece, escribir un sitio web de documentación a mano deja de ser práctico. **Sphinx** es una herramienta que toma tus docstrings (los que ya escribiste) y genera automáticamente páginas HTML con toda la documentación — es lo que usan la mayoría de los paquetes de Python que consultas en internet.

No es necesario que aprendas a usar Sphinx en este curso — alcanza con que sepas que existe y para qué sirve, por si en el futuro necesitas documentar un proyecto más grande. Un ejemplo completo está en `examples/setuptools/sphinx-doc` de este repositorio.

## Testing (pruebas de software)

Una **prueba** (*test*) es un pequeño programa que verifica que otra parte de tu código funciona como esperas — automáticamente, sin que tengas que revisarlo a mano cada vez.

¿Por qué importa?
* **Evita regresiones:** si modificas tu código dentro de 3 meses y sin querer rompes algo que ya funcionaba, las pruebas te avisan de inmediato, en vez de que te enteres semanas después con resultados incorrectos.
* **Facilita colaborar:** si alguien más toca tu código, las pruebas le dicen si algo se rompió.

Tipos de pruebas:
* **Pruebas unitarias:** prueban una función o clase individual, aislada del resto.
* **Pruebas de integración:** prueban cómo varias partes del programa funcionan juntas.

La herramienta más usada hoy en Python es `pytest`.

### Cómo funciona pytest

1. Escribes un archivo que empiece con `test_` (por ejemplo, `test_foobar.py`).
2. Adentro, escribes funciones que también empiecen con `test_`, cada una con al menos un `assert` (una afirmación que debe ser verdadera):
   ```python
   def test_foobar_functionality():
       assert foobar.whatever() == True
   ```
3. Corres el comando `pytest` en la terminal, en esa carpeta.

`pytest` encuentra automáticamente todos los archivos y funciones que sigan ese patrón de nombres, los ejecuta, y te dice cuáles pasaron (✓) y cuáles fallaron (✗) — no tienes que decirle manualmente qué probar.

### Actividad: escribe y corre tus propias pruebas

**Paso 1.** Guarda tu función en un archivo `.py` real (pytest necesita archivos, no celdas de notebook).

In [8]:
%%writefile mis_funciones.py
import numpy as np


def midpoints_array(x):
    """
    Calcula los puntos medios entre elementos consecutivos de x.

    Parameters
    ----------
    x : ndarray
        Arreglo unidimensional con al menos dos elementos.

    Returns
    -------
    ndarray
        Arreglo de len(x) - 1 elementos con los puntos medios.

    Raises
    ------
    ValueError
        Si x tiene menos de dos elementos, porque no hay ningun par
        de vecinos para calcular un punto medio.
    """
    x = np.asarray(x)
    if x.size < 2:
        raise ValueError("Se necesitan al menos dos elementos para calcular puntos medios")
    return 0.5*(x[:-1] + x[1:])


Overwriting mis_funciones.py


**Paso 2.** Escribe un archivo de pruebas para esa función.

In [9]:
%%writefile test_mis_funciones.py
import numpy as np
import pytest
from mis_funciones import midpoints_array

def test_midpoints_dos_elementos():
    resultado = midpoints_array(np.array([0.0, 10.0]))
    assert np.allclose(resultado, [5.0])

def test_midpoints_tamano_correcto():
    x = np.arange(10.0)
    resultado = midpoints_array(x)
    # con 10 elementos de entrada, deberian salir 9 puntos medios
    assert len(resultado) == 9

def test_midpoints_falla_con_lista_vacia():
    # Decidi que un arreglo vacio debe lanzar un error claro,
    # en vez de devolver silenciosamente un arreglo vacio
    with pytest.raises(ValueError):
        midpoints_array(np.array([]))


Overwriting test_mis_funciones.py


**Paso 3.** Ejecuta las pruebas. El `!` al inicio de la celda le dice a Jupyter "esto es un comando de terminal, no Python". Si `pytest` no está instalado, corre antes `!pip install pytest`.

In [10]:
!pytest -v test_mis_funciones.py

============================= test session starts =============================
platform win32 -- Python 3.11.16, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\MSI_COREi5\miniforge3\envs\pyhpc-lite\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\MSI_COREi5\python-for-hpc-clase\notebooks
plugins: anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 3 items

test_mis_funciones.py::test_midpoints_dos_elementos PASSED               [ 33%]
test_mis_funciones.py::test_midpoints_tamano_correcto PASSED             [ 66%]
test_mis_funciones.py::test_midpoints_falla_con_lista_vacia PASSED       [100%]

============================== 3 passed in 0.12s ==============================


Deberías ver `2 passed` en verde al final. Eso significa que ambas pruebas se cumplieron.

**Paso 4 (actividad extra):** agrega una tercera prueba, `test_midpoints_falla_con_lista_vacia`, para el caso de un arreglo vacío `np.array([])`. Primero decide: ¿debería lanzar un error, o devolver un arreglo vacío? Escribe la prueba que confirme el comportamiento que decidiste, y vuelve a correr `pytest -v`.

**Mi decisión para el arreglo vacío**

Decidí que `midpoints_array` debe **lanzar un error** (`ValueError`) cuando recibe menos de dos elementos. Con un arreglo vacío no existe ningún par de vecinos, así que un "punto medio" no tiene sentido.

Si la función devolviera un arreglo vacío sin avisar, el error aparecería después, en otra parte del programa, y sería más difícil de rastrear. Por eso agregué la validación en `mis_funciones.py` y escribí la prueba con `pytest.raises(ValueError)`.

Al volver a correr `pytest -v` obtuve **`3 passed`**: las dos pruebas originales y la nueva.

## Control de versiones con Git

Ya usaste `git` para clonar el repositorio de este curso. `git` sirve para **guardar el historial completo de cambios** de un proyecto: qué cambió, cuándo, y por qué — y te permite volver atrás si algo se rompe.

Consejo: usa `git` para *cualquier* proyecto, incluso uno pequeño y personal — no hace falta que sea un proyecto grande en equipo para que valga la pena.

### Actividad: versiona el paquete que creaste

**Paso 1.** Convierte la carpeta `mi_paquete` en un repositorio de git (en una terminal):
```bash
cd mi_paquete
git init
git add .
git commit -m "Primera version de mi paquete"
```

**Paso 2.** Modifica la función `saludar` en `__init__.py` — por ejemplo, que reciba un segundo parámetro con un mensaje personalizado. Guarda el archivo.

**Paso 3.** Haz un segundo commit:
```bash
git add .
git commit -m "Agrego mensaje personalizado a saludar"
```

**Paso 4.** Corre `git log` y observa los dos commits, cada uno con su propio mensaje.

**Para pensar:** si dentro de un mes se te olvida qué cambiaste y por qué, ¿cómo te ayuda tener estos dos commits separados, en vez de un único commit al final con todo mezclado?

**Mis resultados de la actividad de git**

1. Dentro de `mi_paquete` corrí `git init`, agregué un `.gitignore` (para no versionar `__pycache__/` ni la carpeta `*.egg-info/` que genera `pip`) e hice el primer commit.
2. Modifiqué `saludar` para que reciba un segundo parámetro opcional, `mensaje`, con valor por defecto. Así el código que ya usaba `saludar("Ana")` sigue funcionando igual.
3. Hice el segundo commit. Este es el resultado de `git log`:

```
commit 93ca47733644819ee8517ad263e30f4985d97fd8
Author: David Israel Atilano Quiroz <124050317@upq.edu.mx>
Date:   Tue Nov 3 14:42:31 2026 -0600

    Agrego mensaje personalizado a saludar

commit e251d7abc95f0f396e883713069be5887154d1f2
Author: David Israel Atilano Quiroz <124050317@upq.edu.mx>
Date:   Tue Nov 3 14:42:30 2026 -0600

    Primera version de mi paquete
```

Como el paquete está instalado en modo editable, el cambio funcionó sin reinstalar:
`saludar("Ana", "Bienvenida al curso de HPC.")` devolvió `Hola, Ana! Bienvenida al curso de HPC.`

**Para pensar:** con dos commits separados, cada cambio tiene su propio mensaje que explica qué se hizo. Si dentro de un mes algo falla, con `git log` y `git diff` veo exactamente qué cambió en cada paso, y puedo regresar solo a la versión anterior de `saludar` sin perder lo demás. En un único commit con todo mezclado no sabría qué cambio causó el problema.

### Integración continua (CI): correr las pruebas automáticamente

Plataformas como GitHub y GitLab pueden ejecutar tus pruebas de `pytest` automáticamente, cada vez que subes (`push`) un cambio — sin que tengas que acordarte de correrlas tú mismo. A esto se le llama **integración continua (CI)**.

En GitLab, esto se configura con un archivo `.gitlab-ci.yml` (en GitHub sería `.github/workflows/...`):
```yaml
test-python:
    script:
      - pytest -v
```
Con esto, cada vez que subes cambios, el servidor corre `pytest -v` por ti y te avisa si algo falló — antes de que ese error llegue a afectar a nadie más.

## PyPI y `pip`, en más detalle

Volviendo a la idea del principio: **PyPI** (https://pypi.org) es donde vive casi cualquier paquete público de Python — casi 590 000 paquetes distintos. Cuando corres `pip install numpy`, esto es lo que pasa:

1. `pip` busca el paquete `"numpy"` en PyPI.
2. Descarga el archivo correspondiente (ya "armado", lo que se llama un *wheel*, o el código fuente si hace falta compilarlo).
3. Lo instala en tu entorno actual (¡por eso importa tener el entorno correcto activado!).
4. Si `numpy` a su vez depende de otros paquetes, `pip` los instala también, automáticamente.

`pip uninstall numpy` hace el proceso inverso. Cualquier persona puede subir un paquete a PyPI después de registrarse — no hay un filtro de calidad automático, así que instalar un paquete de un autor desconocido implica cierta confianza.

## Resumen

* Empaquetado, documentación, control de versiones y testing facilitan el desarrollo de software.
* Al principio, muchas veces parecen "overhead" innecesario para un proyecto chico.
* **Pero:** a la larga, cuando el proyecto crece o alguien más necesita usar tu código, esa inversión se paga sola — muchas veces varias veces.

## Autoevaluación

Responde con tus propias palabras (edita esta celda):

1. En una frase, ¿qué es PyPI y qué relación tiene con `pip install`?
2. Tienes una función útil en un notebook de tu tesis. Un compañero de laboratorio la necesita para su propio proyecto. Menciona dos formas de dárselo, y explica por qué empaquetarla es mejor que copiar y pegar el archivo.
3. ¿Cuál es la diferencia entre un comentario y un docstring? ¿Cuándo usarías cada uno?
4. Escribiste una función hace 3 meses, sin pruebas. Ahora la modificas para agregar una funcionalidad nueva. ¿Cómo te ayudaría tener pruebas escritas desde el principio a no romper lo que ya funcionaba?
5. ¿Por qué "funciona en mi computadora" no es suficiente para confiar en que tu código funciona en la computadora de otra persona? ¿Qué herramientas de esta notebook ayudan con ese problema?

**Tus respuestas:**

1. PyPI es el repositorio público en internet donde se publican los paquetes de Python. `pip install` busca ahí el paquete, lo descarga y lo instala en mi entorno junto con sus dependencias.

2. Una forma es mandarle el archivo `.py` para que lo copie a su proyecto. La otra es empaquetar la función (con su `pyproject.toml`) y que la instale con `pip install`, desde una carpeta, desde un repositorio de git o desde PyPI. Empaquetar es mejor porque instala también las dependencias, tiene una versión clara y, si corrijo un error, mi compañero solo actualiza el paquete en lugar de volver a copiar archivos.

3. Un comentario (`#`) es una nota para quien lee o modifica el código, por ejemplo para explicar algo que no es obvio o dejar un pendiente. Un docstring explica cómo se usa una función, clase o módulo (qué recibe y qué devuelve), y se puede consultar con `help()`. Uso comentarios para detalles internos y docstrings para documentar lo que otros van a llamar.

4. Las pruebas guardan cómo debe comportarse la función. Si al agregar la nueva funcionalidad rompo algo que ya funcionaba, al correr `pytest` una prueba falla y me avisa de inmediato, antes de usar resultados incorrectos. Así puedo modificar el código con más confianza.

5. Porque en otra computadora puede haber otra versión de Python o de las librerías, otro sistema operativo o faltar alguna dependencia. Lo vi en esta misma notebook: en Windows tuve que tener cuidado con la carpeta desde donde importaba el paquete. Ayudan los entornos virtuales (`venv`/`conda`) para aislar y reproducir las dependencias, el `pyproject.toml` para declararlas, las pruebas con `pytest` y la integración continua (CI), que corre las pruebas en una máquina limpia cada vez que subo cambios.